
# TROG Prompt Experiments (Qwen3.5-0.8B, 512px resize)

This notebook documents **phase-wise prompt optimizations** for the `trog` (Test for Reception of Grammar) benchmark on **Qwen3.5-0.8B** with images resized to 512px. TROG is a 4-way image multiple-choice task where the model must match an English sentence to the correct image out of four options.

- **99 trials** spanning 33 grammatical trial types (from `noun` to `embedded sentence`).
- **Chance level:** 25%.
- **Artifacts:** per-phase CSVs and logs under `results/trog-phases/`.

## Phases

| Phase | Name | Description |
|-------|------|-------------|
| 0 | Baseline | Current manifest prompts, standard letter parsing |
| 1 | Structured multiline | Sentence on its own line, block option layout (A/B/C/D) |
| 2 | Enhanced parsing | Reverse-scan, "image/option X" patterns, last-letter fallback |
| 3 | System prompt | Role-setting system message + strict answer suffix |
| 4 | Visual grounding | Grammar-type-specific hints (WHO does WHAT, spatial cues) |
| 5 | Sentence decomposition | CoT for complex grammar types (identify elements → match) |


In [1]:
from __future__ import annotations

import csv
import importlib.util
from pathlib import Path

import pandas as pd

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / 'data').is_dir() and (p / 'src').is_dir()),
    Path.cwd().parent.parent,
)

RESULTS = ROOT / "results" / "prompt_optimization" / "trog" / "qwen-0.8b"
# Load prompt builders from the experiment script
_spec = importlib.util.spec_from_file_location(
    "trog_phases", ROOT / "scripts" / "prompt_optimization" / "trog" / "experiment_qwen0.8b.py"
)
tg = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(tg)

manifest_rows = tg.load_manifest(ROOT / "data")
IMAGE_DIR = ROOT / "data" / "assets" / "2026-03-26" / "visual" / "trog"
TRIALS_LIST = tg.build_trials(manifest_rows, IMAGE_DIR)
TRIALS = {t["item_uid"]: t for t in TRIALS_LIST}


def build_prompt(phases: set[int], item_uid: str) -> str:
    """Reconstruct the exact prompt used for a given phase combination."""
    t = TRIALS[item_uid]
    row, phrase = t["row"], t["prompt_phrase"]
    if 1 in phases:
        p = tg.build_prompt_phase1(row, phrase)
    else:
        p = tg.build_prompt_baseline(row, phrase)
    if 4 in phases:
        p = tg.apply_phase4_grounding(p, t["trial_type"])
    if 5 in phases:
        p = tg.apply_phase5_decomposition(p, t["trial_type"])
    if 3 in phases:
        p = tg.apply_phase3_system(p)
    return p


def load_result(csv_name: str, item_uid: str) -> dict | None:
    path = RESULTS / csv_name
    if not path.exists():
        return None
    with open(path, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            if row["item_uid"] == item_uid:
                return row
    return None


## Summary: All Runs

Aggregate accuracy, parse rate, and delta vs baseline across all phase configurations.

In [2]:
PHASE_CSVS = [
    ("phase_baseline.csv", "0 — Baseline"),
    ("phase_1.csv", "1 — Structured multiline"),
    ("phase_2.csv", "2 — Enhanced parsing"),
    ("phase_3.csv", "3 — System prompt"),
    ("phase_4.csv", "4 — Visual grounding"),
    ("phase_5.csv", "5 — Sentence decomposition"),
    ("phase_1_2_3.csv", "1+2+3 — Multiline + parse + sys"),
    ("phase_1_5.csv", "1+5 — Multiline + decomp"),
    ("phase_1_2_3_4_5.csv", "1+2+3+4+5 — All"),
]


def summarize_all():
    rows = []
    baseline_acc = None
    for csv_name, label in PHASE_CSVS:
        path = RESULTS / csv_name
        if not path.exists():
            continue
        with open(path, newline="", encoding="utf-8") as f:
            data = list(csv.DictReader(f))
        n = len(data)
        if n == 0:
            continue
        correct = sum(1 for r in data if r["is_correct"].lower() in ("true", "1"))
        parsed = sum(1 for r in data if r["parsed"].lower() in ("true", "1"))
        acc = correct / n
        pr = parsed / n
        if baseline_acc is None:
            baseline_acc = acc
        delta = "—" if acc == baseline_acc and label.startswith("0") else f"{(acc - baseline_acc)*100:+.1f} pp"
        rows.append({"Phase": label, "N": n, "Accuracy": f"{acc:.1%}", "Parse %": f"{pr:.1%}", "Δ vs baseline": delta})
    return pd.DataFrame(rows)


df_summary = summarize_all()
df_summary.style.hide(axis="index")

Phase,N,Accuracy,Parse %,Δ vs baseline
0 — Baseline,99,36.4%,100.0%,—
1 — Structured multiline,99,39.4%,100.0%,+3.0 pp
2 — Enhanced parsing,99,36.4%,100.0%,+0.0 pp
3 — System prompt,99,41.4%,100.0%,+5.1 pp
4 — Visual grounding,99,37.4%,100.0%,+1.0 pp
5 — Sentence decomposition,99,26.3%,78.8%,-10.1 pp
1+2+3 — Multiline + parse + sys,99,43.4%,100.0%,+7.1 pp
1+5 — Multiline + decomp,99,29.3%,65.7%,-7.1 pp
1+2+3+4+5 — All,99,35.4%,100.0%,-1.0 pp


## Per-Trial-Type Breakdown

Accuracy by grammatical complexity tier for each phase configuration.

In [3]:
def breakdown_by_trial_type(csv_name: str, label: str):
    path = RESULTS / csv_name
    if not path.exists():
        print(f"  {csv_name} not found — skipped")
        return
    with open(path, newline="", encoding="utf-8") as f:
        data = list(csv.DictReader(f))
    by_type: dict[str, dict] = {}
    for r in data:
        tt = r["trial_type"]
        rec = by_type.setdefault(tt, {"N": 0, "Correct": 0, "Parsed": 0})
        rec["N"] += 1
        rec["Correct"] += int(r["is_correct"].lower() in ("true", "1"))
        rec["Parsed"] += int(r["parsed"].lower() in ("true", "1"))
    rows = []
    for tt, rec in sorted(by_type.items(), key=lambda kv: kv[1]["Correct"] / max(kv[1]["N"], 1), reverse=True):
        n = rec["N"]
        rows.append({"Trial Type": tt, "N": n, "Accuracy": f"{rec['Correct']/n:.0%}", "Parse %": f"{rec['Parsed']/n:.0%}"})
    total_n = sum(rec["N"] for rec in by_type.values())
    total_c = sum(rec["Correct"] for rec in by_type.values())
    total_p = sum(rec["Parsed"] for rec in by_type.values())
    rows.append({"Trial Type": "OVERALL", "N": total_n, "Accuracy": f"{total_c/total_n:.0%}", "Parse %": f"{total_p/total_n:.0%}"})
    print(f"\n{'='*60}")
    print(f"  {label}")
    print(f"{'='*60}")
    df = pd.DataFrame(rows)
    print(df.to_string(index=False))


for csv_name, label in PHASE_CSVS:
    breakdown_by_trial_type(csv_name, label)


  0 — Baseline
                         Trial Type  N Accuracy Parse %
                               verb  4     100%    100%
                      gerund phrase  1     100%    100%
     compound preposition condition  1     100%    100%
                               noun  4      75%    100%
                          adjective  4      75%    100%
            two element combination  4      75%    100%
                  reversible active  4      75%    100%
   singular/plural personal pronoun  4      50%    100%
    singular/plural noun inflection  4      50%    100%
                 reversible passive  4      50%    100%
                        disjunctive  4      50%    100%
                    relative clause  5      40%    100%
  advanced conjunction coordinating  3      33%    100%
                           negative  4      25%    100%
          three element combination  4      25%    100%
               comparative/absolute  4      25%    100%
                        X but no

---

## Phase 0 — Baseline

Uses the current manifest prompt format: a single line with the sentence and semicolon-separated image references.

**Example prompt:**
```
Choose the image that matches the text: "the girl is chased by the horse". Answer with A, B, C, or D. A: <image1>; B: <image2>; C: <image3>; D: <image4>
```

**Issues:** Dense single-line format, no visual separation between instruction and options, sentence embedded in a long string.

In [4]:
# Phase 0 example: simple noun
uid0_simple = "trog_noun_shoe"
print("=== PROMPT (Phase 0, noun) ===")
print(build_prompt(set(), uid0_simple))
r = load_result("phase_baseline.csv", uid0_simple)
if r:
    print(f"\n=== MODEL OUTPUT ===")
    print(f"Response: {r['raw_response']}")
    print(f"Predicted: {r['predicted_label']}  Correct: {r['correct_label']}  {'✓' if r['is_correct']=='True' else '✗'}")
else:
    print("  (no results yet — run the experiment first)")

=== PROMPT (Phase 0, noun) ===
Choose the image that matches the text: "shoe". Answer with A, B, C, or D. A: <image1>; B: <image2>; C: <image3>; D: <image4>

=== MODEL OUTPUT ===
Response: C
Predicted: C  Correct: C  ✓


In [5]:
# Phase 0 example: complex (reversible passive)
uid0_complex = "trog_revpassive_girl_chased_horse"
print("=== PROMPT (Phase 0, reversible passive) ===")
print(build_prompt(set(), uid0_complex))
r = load_result("phase_baseline.csv", uid0_complex)
if r:
    print(f"\n=== MODEL OUTPUT ===")
    print(f"Response: {r['raw_response']}")
    print(f"Predicted: {r['predicted_label']}  Correct: {r['correct_label']}  {'✓' if r['is_correct']=='True' else '✗'}")
else:
    print("  (no results yet)")

=== PROMPT (Phase 0, reversible passive) ===
Choose the image that matches the text: "the girl is chased by the horse". Answer with A, B, C, or D. A: <image1>; B: <image2>; C: <image3>; D: <image4>

=== MODEL OUTPUT ===
Response: D
Predicted: D  Correct: A  ✗


---

## Phase 1 — Structured Multiline Prompt

Reformats the prompt into clearly separated lines:
- The sentence is displayed prominently in quotes
- Options A–D are listed one per line
- A clean instruction at the end

**Rationale:** VLMs parse multiline prompts with block-style options more reliably than dense single-line formats, as shown in the EGMA-Math experiments.

In [6]:
# Phase 1 example: reversible passive
uid1 = "trog_revpassive_girl_chased_horse"
print("=== PROMPT (Phase 1) ===")
print(build_prompt({1}, uid1))
r = load_result("phase_1.csv", uid1)
if r:
    print(f"\n=== MODEL OUTPUT ===")
    print(f"Response: {r['raw_response']}")
    print(f"Predicted: {r['predicted_label']}  Correct: {r['correct_label']}  {'✓' if r['is_correct']=='True' else '✗'}")
else:
    print("  (no results yet)")

=== PROMPT (Phase 1) ===
Which image matches the sentence: "the girl is chased by the horse"?

A: <image1>
B: <image2>
C: <image3>
D: <image4>

Answer with one letter.

=== MODEL OUTPUT ===
Response: D
Predicted: D  Correct: A  ✗


---

## Phase 2 — Enhanced Parsing

Adds fallback strategies when the baseline parser fails:
- `"Image A"`, `"Option A"`, `"Picture A"` patterns
- `"I choose/select/pick A"` patterns
- `(A)` parenthetical extraction
- Reverse sentence scan (last sentence containing a label)
- Last standalone letter A–D in the output

**Rationale:** Qwen3.5 may not always output a bare letter; it sometimes wraps the answer in natural language.

In [7]:
uid2 = "trog_relclause_pencil_on_book_that_yellow"
print("=== PROMPT (Phase 2 — same as baseline, parsing differs) ===")
print(build_prompt(set(), uid2))
r_base = load_result("phase_baseline.csv", uid2)
r_p2 = load_result("phase_2.csv", uid2)
if r_base:
    print(f"\n--- Baseline parse ---")
    print(f"  Response: {r_base['raw_response']}")
    print(f"  Predicted: {r_base['predicted_label']}  Parsed: {r_base['parsed']}")
if r_p2:
    print(f"\n--- Phase 2 parse ---")
    print(f"  Response: {r_p2['raw_response']}")
    print(f"  Predicted: {r_p2['predicted_label']}  Parsed: {r_p2['parsed']}")

=== PROMPT (Phase 2 — same as baseline, parsing differs) ===
Choose the image that matches the text: "the pencil is on the book that is yellow". Answer with A, B, C, or D. A: <image1>; B: <image2>; C: <image3>; D: <image4>

--- Baseline parse ---
  Response: C
  Predicted: C  Parsed: True

--- Phase 2 parse ---
  Response: C
  Predicted: C  Parsed: True


---

## Phase 3 — System Prompt + Strict Format

Adds:
- **System prompt:** `"You are a visual reasoning assistant. You match English sentences to the image that best depicts them. Always respond with exactly one letter."`
- **Suffix:** `"Respond with exactly one letter: A, B, C, or D. Nothing else."`

**Rationale:** Guides the model to stay on-task and minimizes verbose responses that may confuse parsing.

In [8]:
uid3 = "trog_3combo_boy_jump_box"
print("=== PROMPT (Phase 3) ===")
print(build_prompt({3}, uid3))
r = load_result("phase_3.csv", uid3)
if r:
    print(f"\n=== MODEL OUTPUT ===")
    print(f"Response: {r['raw_response']}")
    print(f"Predicted: {r['predicted_label']}  Correct: {r['correct_label']}  {'✓' if r['is_correct']=='True' else '✗'}")
else:
    print("  (no results yet)")

=== PROMPT (Phase 3) ===
Choose the image that matches the text: "the boy is jumping over the box". Answer with A, B, C, or D. A: <image1>; B: <image2>; C: <image3>; D: <image4>

Respond with exactly one letter: A, B, C, or D. Nothing else.

=== MODEL OUTPUT ===
Response: D
Predicted: D  Correct: B  ✗


---

## Phase 4 — Visual Grounding Hints

Prepends a grammar-type-specific hint before the prompt to guide the model's attention:

| Grammar category | Hint |
|---|---|
| Reversible active/passive | "Pay close attention to WHO is performing the action and WHO is receiving it." |
| Relative clause / postmodified subject | "Focus on which object or person the description modifies." |
| Embedded sentence | "This sentence has a clause inside another clause. Identify the subject of each part." |
| Spatial types (in/on, above/below) | "Look carefully at the spatial positions of the objects." |
| Logical connectives (X but not Y, neither/nor) | "The sentence specifies what IS and what IS NOT shown. Check both conditions." |

**Rationale:** TROG distractors are specifically designed to catch common grammatical misinterpretations. The hints direct attention to the distinguishing features.

In [9]:
# Phase 4 example: reversible passive
uid4 = "trog_revpassive_elephant_pushed_boy"
print("=== PROMPT (Phase 4) ===")
print(build_prompt({4}, uid4))
r = load_result("phase_4.csv", uid4)
if r:
    print(f"\n=== MODEL OUTPUT ===")
    print(f"Response: {r['raw_response']}")
    print(f"Predicted: {r['predicted_label']}  Correct: {r['correct_label']}  {'✓' if r['is_correct']=='True' else '✗'}")
else:
    print("  (no results yet)")

=== PROMPT (Phase 4) ===
Pay close attention to WHO is performing the action and WHO is receiving it.

Choose the image that matches the text: "the elephant is pushed by the boy". Answer with A, B, C, or D. A: <image1>; B: <image2>; C: <image3>; D: <image4>

=== MODEL OUTPUT ===
Response: C
Predicted: C  Correct: C  ✓


---

## Phase 5 — Sentence Decomposition (CoT)

For complex grammar types, appends step-by-step instructions:
```
Step 1: Identify the key elements in the sentence (who/what, action, recipient/location).
Step 2: Look at each image and check which one matches ALL elements.
Step 3: State your final answer as a single letter.
```

This is only applied to complex types (reversible passive, relative clause, embedded sentence, etc.). Simple types (noun, verb, adjective) are left unchanged.

**Rationale:** Explicit decomposition helps the model avoid jumping to conclusions on grammatically tricky sentences like passives and embedded clauses.

In [10]:
# Phase 5 example: embedded sentence
uid5 = "trog_embedding_book_pencil_on_red"
print("=== PROMPT (Phase 5) ===")
print(build_prompt({5}, uid5))
r = load_result("phase_5.csv", uid5)
if r:
    print(f"\n=== MODEL OUTPUT ===")
    print(f"Response: {r['raw_response']}")
    print(f"Predicted: {r['predicted_label']}  Correct: {r['correct_label']}  {'✓' if r['is_correct']=='True' else '✗'}")
else:
    print("  (no results yet)")

=== PROMPT (Phase 5) ===
Choose the image that matches the text: "the book the pencil is on is red". Answer with A, B, C, or D. A: <image1>; B: <image2>; C: <image3>; D: <image4>

Step 1: Identify the key elements in the sentence (who/what, action, recipient/location).
Step 2: Look at each image and check which one matches ALL elements.
Step 3: State your final answer as a single letter.

=== MODEL OUTPUT ===
Response: D
Predicted: D  Correct: B  ✗


---

## Combined Runs

### Phase 1+2+3 — Multiline + Enhanced Parsing + System Prompt

Combines the three "structural" improvements without the grammar-specific additions.

In [11]:
uid_123 = "trog_revpassive_girl_chased_horse"
print("=== PROMPT (Phase 1+2+3) ===")
print(build_prompt({1, 2, 3}, uid_123))
r = load_result("phase_1_2_3.csv", uid_123)
if r:
    print(f"\n=== MODEL OUTPUT ===")
    print(f"Response: {r['raw_response']}")
    print(f"Predicted: {r['predicted_label']}  Correct: {r['correct_label']}  {'✓' if r['is_correct']=='True' else '✗'}")
else:
    print("  (no results yet)")

=== PROMPT (Phase 1+2+3) ===
Which image matches the sentence: "the girl is chased by the horse"?

A: <image1>
B: <image2>
C: <image3>
D: <image4>

Answer with one letter.

Respond with exactly one letter: A, B, C, or D. Nothing else.

=== MODEL OUTPUT ===
Response: D
Predicted: D  Correct: A  ✗


### Phase 1+5 — Multiline + Sentence Decomposition

Clean prompt format paired with CoT for complex grammar.

In [12]:
uid_15 = "trog_embedding_cat_cow_chase_black"
print("=== PROMPT (Phase 1+5) ===")
print(build_prompt({1, 5}, uid_15))
r = load_result("phase_1_5.csv", uid_15)
if r:
    print(f"\n=== MODEL OUTPUT ===")
    print(f"Response: {r['raw_response']}")
    print(f"Predicted: {r['predicted_label']}  Correct: {r['correct_label']}  {'✓' if r['is_correct']=='True' else '✗'}")
else:
    print("  (no results yet)")

=== PROMPT (Phase 1+5) ===
Which image matches the sentence: "the cat the cow chases is black"?

A: <image1>
B: <image2>
C: <image3>
D: <image4>

Answer with one letter.

Step 1: Identify the key elements in the sentence (who/what, action, recipient/location).
Step 2: Look at each image and check which one matches ALL elements.
Step 3: State your final answer as a single letter.

=== MODEL OUTPUT ===
Response: Step 1: Identify the key elements in the sentence:
- Subject: "the cat"
- Action: "chases"
- Recipient: "the cow"
- Location: "is black"

Step 2: Look at each image and check which one matches ALL elements.
- Image A: Shows a black cat chasing a white cow. The cow is not black.
- Image B: Shows a black cow chasing a white cat. The cat is not black.
- Image C: Shows a white cow and a black cat. The cow is not black.
- Image D: Shows a white cat chasing a black cow. The cow is not black.

Step 3: State your final answer as a single letter.
D
Predicted:   Correct: C  ✗


### Phase 1+2+3+4+5 — All Improvements

Everything combined.

In [13]:
uid_all = "trog_revpassive_horse_chased_man"
print("=== PROMPT (Phase 1+2+3+4+5) ===")
print(build_prompt({1, 2, 3, 4, 5}, uid_all))
r = load_result("phase_1_2_3_4_5.csv", uid_all)
if r:
    print(f"\n=== MODEL OUTPUT ===")
    print(f"Response: {r['raw_response']}")
    print(f"Predicted: {r['predicted_label']}  Correct: {r['correct_label']}  {'✓' if r['is_correct']=='True' else '✗'}")
else:
    print("  (no results yet)")

=== PROMPT (Phase 1+2+3+4+5) ===
Pay close attention to WHO is performing the action and WHO is receiving it.

Which image matches the sentence: "the horse is chased by the man"?

A: <image1>
B: <image2>
C: <image3>
D: <image4>

Answer with one letter.

Step 1: Identify the key elements in the sentence (who/what, action, recipient/location).
Step 2: Look at each image and check which one matches ALL elements.
Step 3: State your final answer as a single letter.

Respond with exactly one letter: A, B, C, or D. Nothing else.

=== MODEL OUTPUT ===
Response: D
Predicted: D  Correct: A  ✗


---

## Analysis: Accuracy by Grammar Complexity Tier

Grouping trial types by linguistic complexity to see where each phase helps most.

In [14]:
SIMPLE_TYPES = {"noun", "verb", "adjective", "two element combination"}
MEDIUM_TYPES = {"negative", "three element combination", "singular/plural personal pronoun",
                "singular/plural noun inflection", "comparative/absolute", "in and on", "above and below"}
COMPLEX_TYPES = set(tg.COMPLEX_GRAMMAR_TYPES)


def tier_accuracy(csv_name: str) -> dict:
    path = RESULTS / csv_name
    if not path.exists():
        return {}
    with open(path, newline="", encoding="utf-8") as f:
        data = list(csv.DictReader(f))
    tiers = {"Simple": (0, 0), "Medium": (0, 0), "Complex": (0, 0)}
    for r in data:
        tt = r["trial_type"]
        correct = int(r["is_correct"].lower() in ("true", "1"))
        if tt in SIMPLE_TYPES:
            c, n = tiers["Simple"]
            tiers["Simple"] = (c + correct, n + 1)
        elif tt in MEDIUM_TYPES:
            c, n = tiers["Medium"]
            tiers["Medium"] = (c + correct, n + 1)
        else:
            c, n = tiers["Complex"]
            tiers["Complex"] = (c + correct, n + 1)
    return {k: v[0]/v[1] if v[1] else 0 for k, v in tiers.items()}


rows = []
for csv_name, label in PHASE_CSVS:
    ta = tier_accuracy(csv_name)
    if ta:
        rows.append({"Phase": label, **{k: f"{v:.0%}" for k, v in ta.items()}})

if rows:
    df_tiers = pd.DataFrame(rows)
    df_tiers.style.hide(axis="index")
else:
    print("No results yet — run the experiment suite first.")

---

## Conclusion

Key takeaways will be filled in after running the experiments:

- Which phase gave the largest overall accuracy boost?
- Which grammar complexity tier benefited most from each optimization?
- Did the combined phases outperform individual ones, or did conflicts arise?
- Recommendations for the production TROG prompt configuration.